# Resolución de E1 — Procesamiento de datos con pandas

Utilizaremos la base de datos `ventas.csv`.

In [2]:
from pathlib import Path

Esto permite la _portabilidad_. Es decir, que otro ordenador pueda encontrar la ruta.

In [3]:
ruta_ventas = Path('../Datos/ventas.csv')
ruta_ventas

PosixPath('../Datos/ventas.csv')

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv(ruta_ventas)

Ya hemos guardado el csv en una variable. Vamos a inspeccionarlo.

In [6]:
print(f"El dataframe tiene {df.shape} filas * columnas")

El dataframe tiene (150, 6) filas * columnas


In [7]:
print("¿Cómo son las columnas del dataframe?")
df.dtypes

¿Cómo son las columnas del dataframe?


fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

Casi todas son un string, excepto el precio, que es un _float_.

Pero, ¿por qué considera que `unidades` es una columna de `str`? Si entramos a `ventas.csv`, vemos que dicha columna es numérica. Esto sucede porque, dado que algunos valores son `NA`, los considera como string, y por tanto toda la columna como string.

In [8]:
for fila in df["unidades"]:
    print(fila)

7
25
1
0
15
22
20
16
18
7
16
2
9
5
6
23
22
5
10
4
9
12
2
2
13
9
17
11
22
20
2
20
10
8
11
16
20
7
12
13
19
4
9
23
3
1
10
7
12
21
10
2
16
2
nan
3
16
24
6
2
22
15
9
nan
13
18
5
17
22
19
1
5
5
4
17
24
19
3
21
24
24
23
22
8
na
19
3
15
-2
15
12
12
10
20
23
4
12
22
13
4
4
16
12
7
14
5
15
11
20
0
23
5
21
22
20
4
12
4
24
17
1
23
24
3
15
9
10
13
11
21
25
25
9
10
3
20
6
9
16
2
24
13
17
20
11
16
19
9
5
8


Lo solucionamos con este comando.

In [9]:
df.replace(["na", "nan"], "", inplace=True)

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027
...,...,...,...,...,...,...
145,2026-02-15,Sur,Sensor A,16,19.50,C027
146,2026-01-13,Oeste,Hub IoT,19,85.00,C040
147,2026-02-09,Oeste,Sensor B,9,32.00,C009
148,2026-01-08,Oeste,Sensor B,5,32.31,C024


In [10]:
for fila in df["unidades"]:
    print(fila)

7
25
1
0
15
22
20
16
18
7
16
2
9
5
6
23
22
5
10
4
9
12
2
2
13
9
17
11
22
20
2
20
10
8
11
16
20
7
12
13
19
4
9
23
3
1
10
7
12
21
10
2
16
2
nan
3
16
24
6
2
22
15
9
nan
13
18
5
17
22
19
1
5
5
4
17
24
19
3
21
24
24
23
22
8

19
3
15
-2
15
12
12
10
20
23
4
12
22
13
4
4
16
12
7
14
5
15
11
20
0
23
5
21
22
20
4
12
4
24
17
1
23
24
3
15
9
10
13
11
21
25
25
9
10
3
20
6
9
16
2
24
13
17
20
11
16
19
9
5
8


In [11]:
print("¿Cuántos valores son NA o NaN en cada columna?")
df.isna().sum()

¿Cuántos valores son NA o NaN en cada columna?


fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64

In [12]:
df.dtypes

fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

Es complicado tratar con un producto que, como ocurre en el csv, no sabemos cuántas unidades se han vendido, o a qué precio.

El primer código da errores, y no es eficiente.

```bash
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    df_valido = pd.DataFrame()
    df_error  = pd.DataFrame()

    i = 0
    j = 0

    for index, row in frame.iterrows():

        condicion_unidades = type(row["unidades"]) == int and row["unidades"] > 0
        condicion_precio_unitario = type(row["precio_unitario"]) == float and \
                                    row["precio_unitario"] > 0
        
        if condicion_unidades and condicion_precio_unitario:
            df_valido[i] = frame[index]
            df_valido[i]["importe"] = frame[index]["unidades"] * frame[index]["precio_unitario"]
            i += 1
        else:
            df_error[j] = frame[index]
            j += 1
            
    print(f"{len(df_valido)} filas válidas y {len(df_error)} filas erróneas")

```

Este sí es correcto (operamos con columnas, y no hace falta ir recorriendo las filas)

In [13]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    unidades numérica y > 0
    precio_unitario numérico y > 0
    columnas derivadas: importe = unidades * precio_unitario solo en válidos (añado una nueva columna) 
    Devuelve (validos, errores).
    """

    #forzamos que la columna sea numérica
    unidades_num = pd.to_numeric(frame["unidades"], errors="coerce")
    precio_num = pd.to_numeric(frame["precio_unitario"], errors="coerce")

    condicion_unidades = (unidades_num > 0) & (unidades_num % 1 == 0)  # enteros positivos
    condicion_precio_unitario = precio_num > 0  # enteros o floats positivos
    condicion = condicion_unidades & condicion_precio_unitario

    df_valido = frame[condicion].copy()
    df_error = frame[~condicion].copy()

    #el df original sigue teniendo strings como na o nan
    df_valido["importe"] = (
        pd.to_numeric(df_valido["unidades"])
        * pd.to_numeric(df_valido["precio_unitario"])
    )
    
    print(f"{len(df_valido)} filas válidas y {len(df_error)} filas erróneas")

    return df_valido, df_error


In [14]:
validar_ventas(df)

140 filas válidas y 10 filas erróneas


(          fecha region   producto unidades  precio_unitario cliente_id  \
 0    2026-01-09    Sur  Cable Kit        7            12.50       C005   
 1    2026-03-14  Oeste    Gateway       25           120.00       C032   
 2    2026-01-26  Oeste    Hub IoT        1            88.96       C016   
 4    2026-02-07  Norte    Hub IoT       15            85.00       C027   
 5    2026-01-11   Este  Cable Kit       22            12.50       C028   
 ..          ...    ...        ...      ...              ...        ...   
 144  2026-03-21  Oeste    Hub IoT       11            85.00       C017   
 145  2026-02-15    Sur   Sensor A       16            19.50       C027   
 146  2026-01-13  Oeste    Hub IoT       19            85.00       C040   
 147  2026-02-09  Oeste   Sensor B        9            32.00       C009   
 148  2026-01-08  Oeste   Sensor B        5            32.31       C024   
 
      importe  
 0      87.50  
 1    3000.00  
 2      88.96  
 4    1275.00  
 5     275.00  
 .

In [15]:
validos, errores = validar_ventas(df)

140 filas válidas y 10 filas erróneas


#### Vemos las ventas totales por región

In [21]:
validos.groupby("region")["importe"].sum()

region
Este     33122.53
Norte    22914.63
Oeste    19164.66
Sur      21396.35
Name: importe, dtype: float64

Lo ordenamos

In [22]:
validos.groupby("region")["importe"].sum().sort_values(ascending=False)

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64

#### Vemos el top 3 de productos según ventas

Este es el total de importe por cada producto

In [24]:
validos.groupby("producto")["importe"].sum()

producto
Cable Kit     4901.08
Gateway      40440.00
Hub IoT      33058.66
Sensor A      6629.77
Sensor B     11568.66
Name: importe, dtype: float64

El importe total que se ha vendido es:

In [49]:
float(validos.groupby("producto")["importe"].sum().sum())

96598.17000000001

Los productos que más han vendido son: Gateway, Hub IoT y Sensor B

In [29]:
validos.groupby("producto")["importe"].sum().sort_values(ascending=False)[:3]

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64

Estas son las compras de los distintos clientes

In [40]:
validos.groupby("cliente_id")["unidades"].sum().reset_index()

,cliente_id,unidades
0,C001,94
1,C002,2151024
2,C003,201220
3,C004,13313181713219
4,C005,718102420
5,C006,19
6,C007,1621
7,C009,12242214169
8,C010,73116
9,C011,23242


In [44]:
ventas_por_id = validos.groupby("cliente_id")["unidades"].sum().reset_index()
ventas_por_id["unidades"] = pd.to_numeric(ventas_por_id["unidades"], errors="coerce")

In [45]:
ventas_por_id[ventas_por_id["unidades"] > 1]

,cliente_id,unidades
0,C001,94
1,C002,2151024
2,C003,201220
3,C004,13313181713219
4,C005,718102420
5,C006,19
6,C007,1621
7,C009,12242214169
8,C010,73116
9,C011,23242


In [46]:
validos.to_csv("../Datos/ventas_limpias.csv")

In [ ]:
filas_totales = len(df)
filas_validas = len(validos)
filas_invalidas = len(errores)
importe_total = float(validos.groupby("producto")["importe"].sum().sum())

In [50]:
import json

In [53]:
ruta_json = Path("../Datos/calidad_datos.json")

metricas = {
    "filas_totales": len(df),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": float(validos.groupby("producto")["importe"].sum().sum()),
}

with open(ruta_json, "w", encoding="utf-8") as archivo:
    json.dump(metricas, archivo, ensure_ascii=False, indent=4)